In [13]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

## PreRequisites

In [14]:
import pandas as pd

from radp.digital_twin.utils.gis_tools import GISTools
from notebooks.radp_library import calculate_received_power

In [15]:
simple_ue = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/topology.csv')

In [16]:
simple_ue.drop(columns=['mock_ue_id', 'tick'], inplace=True)
simple_ue

,longitude,latitude
0,-22.625309,59.806764
1,119.764151,54.857584
2,72.095437,-20.253892
3,-67.548009,-38.100941
4,59.867089,-83.103930
...,...,...
1995,45.564260,42.609846
1996,132.457280,17.241235
1997,-101.217659,72.295988
1998,-16.480045,-26.656397


## Functions

In [17]:
def f0(data,topology):
    if topology["cell_id"].dtype == object:
            topology["cell_id"] = (
                topology["cell_id"].str.replace("cell_", "").astype(int)
            )
    data["key"] = 1
    topology["key"] = 1
    combined_df = pd.merge(data, topology, on="key").drop("key", axis=1)
    return combined_df

In [18]:
def f1(cartesian_df):
    cartesian_df["log_distance"] = cartesian_df.apply(
        lambda row: GISTools.get_log_distance(
            row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
        ),
        axis=1,
    )
    return cartesian_df

In [19]:
def f2(cartesian_df):
    cartesian_df["cell_rxpwr_dbm"] = cartesian_df.apply(
        lambda row: calculate_received_power(
            row["log_distance"], row["cell_carrier_freq_mhz"]
        ),
        axis=1,
    )
    return cartesian_df

In [20]:
def f3(cartesian_df):
    cartesian_df["relative_bearing"] = cartesian_df.apply(
        lambda row: GISTools.get_relative_bearing(
            row["cell_az_deg"],
            row["cell_lat"],
            row["cell_lon"],
            row["latitude"],
            row["longitude"],
        ),
        axis=1,
    )
    return cartesian_df

In [ ]:
def preprocess_ue_data(data, topology):
    cartesian_df = f0(data, topology)
    cartesian_df = f1(cartesian_df)
    return f2(cartesian_df)


## Bebugging

In [22]:
cartesian_df = f0(simple_ue, topology)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100
3,119.764151,54.857584,35.690556,139.691944,1,0,2100
4,119.764151,54.857584,35.690556,139.691944,2,120,2100
...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100


In [23]:
cartesian_df = f1(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124
...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100,16.681342
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100,16.681342
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100,15.788136
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100,15.788136


In [24]:
cartesian_df = f2(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691,-100.000472
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691,-100.000472
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691,-100.000472
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124,-99.287948
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124,-99.287948
...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100,16.681342,-100.339006
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100,16.681342,-100.339006
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100,15.788136,-99.861003
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100,15.788136,-99.861003


In [25]:
cartesian_df = f3(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm,relative_bearing
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691,-100.000472,351.153872
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691,-100.000472,231.153872
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691,-100.000472,111.153872
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124,-99.287948,330.617727
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124,-99.287948,210.617727
...,...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100,16.681342,-100.339006,167.318054
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100,16.681342,-100.339006,47.318054
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100,15.788136,-99.861003,332.079388
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100,15.788136,-99.861003,212.079388


In [ ]:
preprocess_ue_data(simple_ue, topology)

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691,-100.000472
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691,-100.000472
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691,-100.000472
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124,-99.287948
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124,-99.287948
...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100,16.681342,-100.339006
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100,16.681342,-100.339006
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100,15.788136,-99.861003
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100,15.788136,-99.861003
